# AIRPATH-AI — Milestone 2B XGBoost forecasting

This notebook presents the frozen validation and locked-test outputs. It intentionally does **not** rerun model selection or test evaluation when executed. The one-shot experiment entry point is `python3 -m src.xgboost_forecasting`.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.xgboost_forecasting import HourlyStationForecaster

TABLES = ROOT / "reports" / "tables"
MODELS = ROOT / "data" / "processed" / "models"

In [2]:
metadata = json.loads((MODELS / "metadata.json").read_text())
metrics = pd.read_csv(TABLES / "xgboost_metrics.csv")
search = pd.read_csv(TABLES / "xgboost_validation_search.csv")
improvements = pd.read_csv(TABLES / "xgboost_improvement_over_persistence.csv")
imputation = pd.read_csv(TABLES / "xgboost_weather_imputation.csv")
importance = pd.read_csv(TABLES / "xgboost_feature_importance.csv")
shap_importance = pd.read_csv(TABLES / "xgboost_shap_importance.csv")

display(metadata)
display(metrics.loc[metrics["Station_No"].astype(str).eq("ALL")])

{'selected_version': 'v1',
 'horizons_hours': [1, 2, 3],
 'station_ids': ['1', '2', '3', '4', '5', '6'],
 'random_seed': 42,
 'selected_parameters': {'1': {'n_estimators': 250,
   'max_depth': 3,
   'learning_rate': 0.05,
   'subsample': 0.9,
   'colsample_bytree': 0.9,
   'min_child_weight': 1},
  '2': {'n_estimators': 250,
   'max_depth': 3,
   'learning_rate': 0.05,
   'subsample': 0.9,
   'colsample_bytree': 0.9,
   'min_child_weight': 1},
  '3': {'n_estimators': 150,
   'max_depth': 3,
   'learning_rate': 0.1,
   'subsample': 1.0,
   'colsample_bytree': 0.9,
   'min_child_weight': 1}},
 'time_resolution': 'hourly',
 'spatial_support': 'observed HealthyAir stations only'}

,model,split,Station_No,horizon_hours,n,mae,rmse,r2
0,persistence,validation,ALL,ALL,25744,4.486610,9.218151,0.399007
1,persistence,validation,ALL,1,8603,3.038068,7.025280,0.651794
2,persistence,validation,ALL,2,8580,4.639194,9.453301,0.367451
3,persistence,validation,ALL,3,8561,5.789336,10.787420,0.175577
28,persistence,test,ALL,ALL,23655,4.316185,8.746862,0.417159
29,persistence,test,ALL,1,7929,2.999557,7.140499,0.613373
30,persistence,test,ALL,2,7883,4.494094,8.876511,0.399070
31,persistence,test,ALL,3,7843,5.468435,10.000589,0.235379
56,historical_time,validation,ALL,ALL,25744,8.266889,11.247184,0.105317
57,historical_time,validation,ALL,1,8603,8.271190,11.258258,0.105766


In [3]:
display(search.sort_values(["horizon_hours", "validation_mae"]))
display(improvements.loc[
    improvements["Station_No"].astype(str).eq("ALL")
])
display(imputation)
display(importance.groupby("horizon_hours", group_keys=False).head(10))
display(shap_importance.groupby("horizon_hours", group_keys=False).head(10))

,horizon_hours,candidate_id,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,validation_n,validation_mae,validation_rmse,validation_r2
1,1,2,250,3,0.05,0.9,0.9,1,8603,4.698592,8.088267,0.538449
0,1,1,150,3,0.10,1.0,0.9,1,8603,4.712234,8.102410,0.536833
2,1,3,200,4,0.05,0.9,0.9,1,8603,4.851261,8.327629,0.510727
3,1,4,250,5,0.05,0.8,0.8,3,8603,4.897351,8.513147,0.488684
5,2,2,250,3,0.05,0.9,0.9,1,8580,5.676846,9.007634,0.425687
4,2,1,150,3,0.10,1.0,0.9,1,8580,5.800866,9.080030,0.416418
7,2,4,250,5,0.05,0.8,0.8,3,8580,5.882833,9.364574,0.379269
6,2,3,200,4,0.05,0.9,0.9,1,8580,5.890084,9.203838,0.400395
8,3,1,150,3,0.10,1.0,0.9,1,8561,6.359879,9.646137,0.340793
9,3,2,250,3,0.05,0.9,0.9,1,8561,6.360099,9.633372,0.342537


,model,split,Station_No,horizon_hours,n,mae,rmse,r2,persistence_mae,persistence_rmse,persistence_r2,mae_absolute_improvement,mae_percent_improvement,rmse_absolute_improvement,rmse_percent_improvement,r2_absolute_improvement
0,xgboost_v1,test,ALL,ALL,23655,5.172645,8.286695,0.476872,4.316185,8.746862,0.417159,-0.856460,-19.842975,0.460167,5.260938,0.059713
1,xgboost_v1,test,ALL,1,7929,4.433343,7.526112,0.570487,2.999557,7.140499,0.613373,-1.433786,-47.799923,-0.385614,-5.400372,-0.042886
2,xgboost_v1,test,ALL,2,7883,5.273807,8.352866,0.467879,4.494094,8.876511,0.399070,-0.779713,-17.349721,0.523646,5.899231,0.068809
3,xgboost_v1,test,ALL,3,7843,5.818376,8.928687,0.390505,5.468435,10.000589,0.235379,-0.349941,-6.399289,1.071901,10.718382,0.155126
28,xgboost_v1,validation,ALL,ALL,25744,5.577077,8.935651,0.435279,4.486610,9.218151,0.399007,-1.090466,-24.304909,0.282500,3.064603,0.036272
29,xgboost_v1,validation,ALL,1,8603,4.698592,8.088267,0.538449,3.038068,7.025280,0.651794,-1.660524,-54.657237,-1.062987,-15.130879,-0.113345
30,xgboost_v1,validation,ALL,2,8580,5.676846,9.007634,0.425687,4.639194,9.453301,0.367451,-1.037652,-22.367077,0.445668,4.714414,0.058236
31,xgboost_v1,validation,ALL,3,8561,6.359879,9.646137,0.340793,5.789336,10.787420,0.175577,-0.570544,-9.855081,1.141284,10.579764,0.165216
56,xgboost_v2,test,ALL,ALL,23655,5.269298,8.423674,0.459434,4.316185,8.746862,0.417159,-0.953112,-22.082281,0.323187,3.694896,0.042275
57,xgboost_v2,test,ALL,1,7929,4.520634,7.646231,0.556667,2.999557,7.140499,0.613373,-1.521077,-50.710071,-0.505732,-7.082592,-0.056706


,fit_stage,horizon_hours,fit_rows,temperature_missing,humidity_missing,temperature_training_median,humidity_training_median
0,validation_fit,1,34235,2106,2106,28.093333,71.280000
1,validation_fit,2,33994,2101,2101,28.088333,71.266667
2,validation_fit,3,33772,2097,2097,28.083333,71.282609
3,final_test_fit,1,42838,3664,3664,28.035093,70.301681
4,final_test_fit,2,42574,3657,3657,28.030000,70.301667
5,final_test_fit,3,42333,3651,3651,28.022684,70.305833


,model,horizon_hours,feature,importance
0,xgboost_v1,1,history_time__pm25_lag_1h,0.659261
1,xgboost_v1,1,history_time__pm25_lag_2h,0.053245
2,xgboost_v1,1,history_time__hour,0.048672
3,xgboost_v1,1,history_time__pm25_lag_3h,0.045219
4,xgboost_v1,1,history_time__month,0.032987
5,xgboost_v1,1,station__Station_No_4,0.032967
6,xgboost_v1,1,history_time__day_of_week,0.029401
7,xgboost_v1,1,station__Station_No_3,0.026124
8,xgboost_v1,1,station__Station_No_1,0.025103
9,xgboost_v1,1,station__Station_No_6,0.023879


,model,horizon_hours,feature,mean_absolute_shap,shap_sample_rows
0,xgboost_v1,1,history_time__pm25_lag_1h,6.242698,500
1,xgboost_v1,1,history_time__hour,0.984003,500
2,xgboost_v1,1,history_time__month,0.688441,500
3,xgboost_v1,1,station__Station_No_4,0.428445,500
4,xgboost_v1,1,history_time__pm25_lag_2h,0.266533,500
5,xgboost_v1,1,station__Station_No_3,0.256881,500
6,xgboost_v1,1,history_time__pm25_lag_3h,0.250136,500
7,xgboost_v1,1,history_time__day_of_week,0.225743,500
8,xgboost_v1,1,station__Station_No_5,0.207499,500
9,xgboost_v1,1,station__Station_No_1,0.049273,500


## Target-time API

The serialized `HourlyStationForecaster` exposes `predict_pm25(station_or_location, target_time, *, prediction_time, pm25_lags, temperature=None, humidity=None)`. It supports only known monitored stations and exact 1–3 hour targets. Geographic road locations and sub-hourly times are rejected until separately validated spatial and higher-resolution layers exist.

In [4]:
forecaster = HourlyStationForecaster.load(
    MODELS / "hourly_station_forecaster.joblib"
)
print(type(forecaster).__name__)
print("Selected version:", forecaster.version)
print("Supported monitored stations:", forecaster.station_ids)
print("Supported horizons:", sorted(forecaster.models))

HourlyStationForecaster
Selected version: v1
Supported monitored stations: ('1', '2', '3', '4', '5', '6')
Supported horizons: [1, 2, 3]


The locked test results must not be used for further tuning. No XGBoost experiment is rerun from this presentation notebook.